In [1]:
import os
import pyam
import pandas as pd
from pathlib import Path

<IPython.core.display.Javascript object>

In [2]:
IKEA_ISOS = ["DEU","POL","BRA","MEX","KEN","MAR","MOZ",#"EU27",
             "NGA","SEN","ZAF","USA","NAM","DZA","TUR","SAU",
             "ARE","BGD","IND","IDN","PAK","VNM","GBR","AUS","CHN","JPN"]

IKEA_ISOS.sort()

In [3]:
import datatoolbox as dt

## Load data

In [4]:
BOX_MOUNT_PATH = Path("~/Library/CloudStorage/Box-Box").expanduser()
if not BOX_MOUNT_PATH.is_dir():
    BOX_MOUNT_PATH = Path("~/Box").expanduser()

In [5]:
DSCALE_PATH = (
    "Climate Policy Team/"
    "02 - Projects/"
    "IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/"
    "WP2 - National 1.5°C Pathways/"
    "Downscaling/"
    "DSCALE"
)

DSCALE_PATH: os.PathLike = BOX_MOUNT_PATH / DSCALE_PATH

In [6]:
REMIND_PATH = (
    "Climate Policy Team/"
    "02 - Projects/"
    "IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/"
    "WP1 - Updating global 1.5°C Pathways/"
    "1 - PIK/"
    "Scenario Data/"
    "SUBMITTED_SCENARIOS_August2025/"

)

REMIND_PATH: os.PathLike = BOX_MOUNT_PATH / REMIND_PATH

In [7]:
NPE_PATH = (
    "Climate Policy Team/"
    "02 - Projects/"
    "IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/"
    "WP2 - National 1.5°C Pathways/"
    "Country assessment"

)

NPE_PATH: os.PathLike = BOX_MOUNT_PATH / NPE_PATH

In [8]:
harm_remind = pyam.IamDataFrame(DSCALE_PATH / "data/step2_project_folder/snapshot_v1/npe_core_scenario_harmonised_AR6_reporting.csv")

pyam - INFO: Running in a notebook, setting up a basic logging at level INFO
pyam.core - INFO: Reading file /Users/marie-charlottegeffray/Library/CloudStorage/Box-Box/Climate Policy Team/02 - Projects/IKEA NDC 1.5° Pathways 23-25 - phase II/2 - Work Packages/WP2 - National 1.5°C Pathways/Downscaling/DSCALE/data/step2_project_folder/snapshot_v1/npe_core_scenario_harmonised_AR6_reporting.csv


In [9]:
siamese = pyam.IamDataFrame(NPE_PATH / "3_Data/results_v13staging/siamese.csv")

pyam.core - INFO: Reading file /Users/marie-charlottegeffray/Library/CloudStorage/Box-Box/Climate Policy Team/02 - Projects/IKEA NDC 1.5° Pathways 23-25 - phase II/2 - Work Packages/WP2 - National 1.5°C Pathways/Country assessment/3_Data/results_v13staging/siamese.csv


## Macro-regions countries - Clean REMIND data

In [10]:
harm_remind_countries=harm_remind.filter(region = ["USA", "IND", "CHA", "JPN"])
harm_remind_countries=harm_remind_countries.rename(region = {"CHA":"CHN"})
harm_remind_countries=harm_remind_countries.filter(year = [2023, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070])

In [11]:
# Renaming variables
harm_remind_countries.rename(variable = {x: x.replace("Oil", "Oil and Other") for x in harm_remind_countries.filter(variable = "Final Energy|*|Oil").variable}, inplace=True)
harm_remind_countries.rename(variable = {x: x.replace("Natural Gas", "Natural gas") for x in harm_remind_countries.filter(variable = "Final Energy|*|Natural Gas").variable}, inplace=True)

In [12]:
harm_remind_countries.aggregate(
    'Primary Energy|Non-Biomass Renewables', 
    [
        'Primary Energy|Solar',
        'Primary Energy|Wind', 
        'Primary Energy|Geothermal',
        'Primary Energy|Hydro',

    ], append=True
)

In [13]:
# Creating variables
# harm_remind_countries.subtract('Primary Energy|Oil|w/o CCS', 'Primary Energy|Oil', 'Primary Energy|Oil|w/ CCS', append=True)
# harm_remind_countries.subtract('Secondary Energy|Electricity|Oil|w/o CCS', 'Secondary Energy|Electricity|Oil', 'Secondary Energy|Electricity|Oil|w/ CCS', append=True)

In [14]:
# # Computing variable Other
# harm_remind_countries.aggregate("Final Energy|Industry|SubTotal", [
#     'Final Energy|Industry|Electricity',
#     # 'Final Energy|Industry|Gases',
#     'Final Energy|Industry|Gases|Biomass',
#     'Final Energy|Industry|Gases|Electricity',
#     'Final Energy|Industry|Gases|Hydrogen synfuel',
#     'Final Energy|Industry|Gases|Natural Gas',
#     'Final Energy|Industry|Heat',
#     'Final Energy|Industry|Hydrogen',
#     # 'Final Energy|Industry|Liquids',
#     'Final Energy|Industry|Liquids|Biomass',
#     'Final Energy|Industry|Liquids|Electricity',
#     'Final Energy|Industry|Liquids|Oil and Other',
#     # 'Final Energy|Industry|Solids',
#     'Final Energy|Industry|Solids|Biomass',
#     'Final Energy|Industry|Solids|Coal'], 
#     append = True)

# harm_remind_countries.subtract("Final Energy|Industry", "Final Energy|Industry|SubTotal", "Final Energy|Industry|Other", append=True)
# harm_remind_countries.filter(variable = "*SubTotal", keep=False, inplace=True)

In [15]:
# harm_remind_countries.aggregate("Final Energy|Residential and Commercial|SubTotal", [
#     'Final Energy|Residential and Commercial|Electricity',
# #  'Final Energy|Residential and Commercial|Gases',
#  'Final Energy|Residential and Commercial|Gases|Biomass',
#  'Final Energy|Residential and Commercial|Gases|Electricity',
#  'Final Energy|Residential and Commercial|Gases|Natural gas',
#  'Final Energy|Residential and Commercial|Heat',
#  'Final Energy|Residential and Commercial|Hydrogen',
# #  'Final Energy|Residential and Commercial|Liquids',
#  'Final Energy|Residential and Commercial|Liquids|Biomass',
#  'Final Energy|Residential and Commercial|Liquids|Electricity',
#  'Final Energy|Residential and Commercial|Liquids|Oil and Other',
# #  'Final Energy|Residential and Commercial|Solids',
#  'Final Energy|Residential and Commercial|Solids|Biomass',
#  'Final Energy|Residential and Commercial|Solids|Coal'], 
#     append = True)

# harm_remind_countries.subtract("Final Energy|Residential and Commercial", "Final Energy|Residential and Commercial|SubTotal", "Final Energy|Residential and Commercial|Other", append=True)
# harm_remind_countries.filter(variable = "*SubTotal", keep=False, inplace=True)

In [16]:
# harm_remind_countries.aggregate("Final Energy|Transportation|SubTotal", [
#     'Final Energy|Transportation|Electricity',
# #  'Final Energy|Transportation|Gases',
#  'Final Energy|Transportation|Gases|Biomass',
#  'Final Energy|Transportation|Gases|Electricity',
#  'Final Energy|Transportation|Gases|Natural gas',
#  'Final Energy|Transportation|Hydrogen',
# #  'Final Energy|Transportation|Liquids',
#  'Final Energy|Transportation|Liquids|Biomass',
#  'Final Energy|Transportation|Liquids|Coal',
#  'Final Energy|Transportation|Liquids|Electricity',
#  'Final Energy|Transportation|Liquids|Natural gas',
#  'Final Energy|Transportation|Liquids|Oil and Other'], 
#     append = True)

# harm_remind_countries.subtract("Final Energy|Transportation", "Final Energy|Transportation|SubTotal", "Final Energy|Transportation|Other", append=True)
# harm_remind_countries.filter(variable = "*SubTotal", keep=False, inplace=True)

## DSCALE Countries

In [17]:
results_folder = "../results/5_Explorer_and_New_Variables"
config_file_name="REMIND_Q1_2026"
file_suffix = "09_03_2026"

In [18]:
step5h_energy_consumptions = pyam.IamDataFrame(f"{results_folder}/{config_file_name}/{file_suffix}/REMIND 3.4_2023_harmo_step5h_hydrogen.csv")

pyam.core - INFO: Reading file ../results/5_Explorer_and_New_Variables/REMIND_Q1_2026/09_03_2026/REMIND 3.4_2023_harmo_step5h_hydrogen.csv


In [19]:
step5f_emissions = pd.read_csv(f"{results_folder}/{config_file_name}/{file_suffix}/Emissions_{file_suffix}.csv", index_col = [0,1,2,3,4])
step5f_emissions.index.names = ['model', 'region', 'scenario', 'variable', 'unit']
step5f_emissions = pyam.IamDataFrame(step5f_emissions)

In [20]:
dscale_results = pyam.concat([
    step5f_emissions, 
    step5h_energy_consumptions.filter(variable = ["Emissions*", "*Carbon*"], keep=False)
])

dscale_results=dscale_results.filter(region = ["CHN", "USA", "IND", "JPN"], keep=False)

In [21]:
# Renaming variables
dscale_results.rename(variable = {x: x.replace("Oil", "Oil and Other") for x in dscale_results.filter(variable = "Final Energy|*|Oil").variable}, inplace=True)
dscale_results.rename(variable = {x: x.replace("Natural Gas", "Natural gas") for x in dscale_results.filter(variable = "Final Energy|*|Natural Gas").variable}, inplace=True)

pyam.core - WARNING: Filtered IamDataFrame is empty!
pyam.core - WARNING: Filtered IamDataFrame is empty!


In [22]:
# Creating variables
# dscale_results.subtract('Primary Energy|Oil|w/o CCS', 'Primary Energy|Oil', 'Primary Energy|Oil|w/ CCS', append=True)
# dscale_results.subtract('Secondary Energy|Electricity|Oil|w/o CCS', 'Secondary Energy|Electricity|Oil', 'Secondary Energy|Electricity|Oil|w/ CCS', append=True)

In [23]:
gas_coal = (
    dscale_results
    .filter(variable = ["Secondary Energy|Electricity|Gas", "Secondary Energy|Electricity|Coal"])
    .rename(variable = {
            "Secondary Energy|Electricity|Gas":"Secondary Energy|Electricity|Gas|w/o CCS", 
            "Secondary Energy|Electricity|Coal":"Secondary Energy|Electricity|Coal|w/o CCS"})
    )

In [24]:
dscale_results = pyam.concat([
    dscale_results, 
    gas_coal
])

In [25]:
# Add EU27
dscale_results.aggregate_region(
    variable = dscale_results.variable, 
    region = "EU27", 
    subregions=list(dt.REGIONS.EU27), append = True)

In [26]:
# Add RE for Primary Energy
dscale_results.aggregate('Primary Energy|Non-Biomass Renewables', [
    'Primary Energy|Solar', 
    'Primary Energy|Wind', 
    'Primary Energy|Hydro', 
    'Primary Energy|Geothermal'
], append=True)

In [27]:
dscale_results.filter(variable = "Secondary Energy|Electricity*", region = "BRA").variable

['Secondary Energy|Electricity',
 'Secondary Energy|Electricity|Biomass',
 'Secondary Energy|Electricity|Coal',
 'Secondary Energy|Electricity|Coal|w/o CCS',
 'Secondary Energy|Electricity|Gas',
 'Secondary Energy|Electricity|Gas|w/o CCS',
 'Secondary Energy|Electricity|Geothermal',
 'Secondary Energy|Electricity|Hydro',
 'Secondary Energy|Electricity|Hydrogen',
 'Secondary Energy|Electricity|Nuclear',
 'Secondary Energy|Electricity|Oil',
 'Secondary Energy|Electricity|Solar',
 'Secondary Energy|Electricity|T&D losses, conversion to synthetic fuels and net exports',
 'Secondary Energy|Electricity|Wind']

In [28]:
dscale_results.filter(variable = "Primary Energy|*", region = "BRA").variable

['Primary Energy|Biomass',
 'Primary Energy|Coal',
 'Primary Energy|Coal|w/ CCS',
 'Primary Energy|Coal|w/o CCS',
 'Primary Energy|Fossil',
 'Primary Energy|Fossil|w/ CCS',
 'Primary Energy|Fossil|w/o CCS',
 'Primary Energy|Gas',
 'Primary Energy|Gas|w/ CCS',
 'Primary Energy|Gas|w/o CCS',
 'Primary Energy|Geothermal',
 'Primary Energy|Hydro',
 'Primary Energy|Non-Biomass Renewables',
 'Primary Energy|Nuclear',
 'Primary Energy|Oil',
 'Primary Energy|Oil|w/ CCS',
 'Primary Energy|Oil|w/o CCS',
 'Primary Energy|Solar',
 'Primary Energy|Wind']

## Merge MACRO COUNTRIES and DSCALE COUNTRIES

In [29]:
npe_results = pyam.concat([
    dscale_results, 
    harm_remind_countries
])

In [30]:
# Filtering out unnecessary variables
npe_results=npe_results.filter(variable = siamese.variable + [
    'Secondary Energy|Electricity|Solar', 
    'Secondary Energy|Electricity|Wind', 
    'Secondary Energy|Electricity|Hydro', 
    'Secondary Energy|Electricity|Geothermal',
    'Primary Energy|Solar', 
    'Primary Energy|Wind', 
    'Primary Energy|Hydro', 
    'Primary Energy|Geothermal',
    'Primary Energy|Hydrogen',
    'Primary Energy|Biomass',
    'Final Energy|Industry*',
    'Final Energy|Residential*',
    'Final Energy|Transportation*', 
    ])

In [31]:
# Renaming the units
npe_results.rename(unit = {
    'Mt CH4/yr': 'Mt CH4 / yr',
    'Mt CO2/yr':'Mt CO2 / yr',
    'Mt CO2-equiv/yr':'Mt CO2eq/yr',
    'kt N2O/yr':'kt N2O / yr', 
    'EJ/yr':'EJ / yr',
    'million':'millions',
}, inplace=True)

In [32]:
npe_results=npe_results.filter(unit = "%", keep=False)

## Add historical data

In [33]:
from data_shepherd import emissions, energy_forms, utils

In [34]:
siam = utils.Pathways(
    siamese.filter(model = ["SIAMESE", "MESSAGE*", "REMIND*", "OECD_Env-Growth"]), 
    siamese.filter(model = ["SIAMESE", "MESSAGE*", "REMIND*", "OECD_Env-Growth"], keep=False))

In [35]:
emissions_data = emissions.get_historic_iea(source = "IEA_GHG_FUEL_DETAILED_2025")

/Users/marie-charlottegeffray/opt/anaconda3/envs/1o5/lib/python3.9/site-packages/numpy/core/fromnumeric.py:84: FutureWarning: In a future version, DataFrame.max(axis=None) will return a scalar max over the entire DataFrame. To retain the old behavior, use 'frame.max(axis=0)' or just 'frame.max()'
  return reduction(axis=axis, out=out, **passkwargs)
/Users/marie-charlottegeffray/opt/anaconda3/envs/1o5/lib/python3.9/site-packages/numpy/core/fromnumeric.py:84: FutureWarning: In a future version, DataFrame.max(axis=None) will return a scalar max over the entire DataFrame. To retain the old behavior, use 'frame.max(axis=0)' or just 'frame.max()'
  return reduction(axis=axis, out=out, **passkwargs)
data_shepherd.emissions - WARNING: Using Energy|Supply|Electricity and Heat timeseries for Electricity for CAN, DNK, IRL, ISL, JPN, NZL, POL


In [36]:
energy = pyam.IamDataFrame("../input_data/input_reference_iea_2022.csv")
energy = pyam.IamDataFrame(
    energy.timeseries()
    .rename(index = {
        "IEA":"IEA_WEB_DETAILED_2025", 
        "Historic data":"Historic"
    })
)

pyam.core - INFO: Reading file ../input_data/input_reference_iea_2022.csv


In [37]:
energy = pyam.IamDataFrame(energy.timeseries().replace(1e-9, 0))

In [38]:
hist = pyam.concat([
    energy, 
    emissions_data
])

hist.filter(variable = 
            siam.hist.variable + [
                'Secondary Energy|Electricity|Solar', 
                'Secondary Energy|Electricity|Wind', 
                'Secondary Energy|Electricity|Hydro', 
                'Secondary Energy|Electricity|Geothermal',
                'Final Energy|Industry*',
                'Final Energy|Residential*',
                'Final Energy|Transportation*'], 
            inplace=True)

hist.filter(region=siam.hist.region, inplace=True)

In [39]:
# Renaming variables
hist.rename(variable = {x: x.replace("Oil", "Oil and Other") for x in hist.filter(variable = "Final Energy|*|Oil").variable}, inplace=True)
hist.rename(variable = {x: x.replace("Natural Gas", "Natural gas") for x in hist.filter(variable = "Final Energy|*|Natural Gas").variable}, inplace=True)

In [40]:
# Renaming the units
hist.rename(unit = {
    'Mt CO2/yr':'Mt CO2 / yr',
    'EJ/yr':'EJ / yr',
}, inplace=True)

In [41]:
hist.aggregate_region("*", region = "EU27", subregions=dt.REGIONS.EU27, append=True)

In [42]:
hist=pyam.IamDataFrame(hist.timeseries())

In [43]:
npe_results_hist = pyam.concat([npe_results, hist])

In [44]:
npe_results_hist.filter(variable = [
    "Emissions|CH4*", 
    "Emissions|N2O*", 
    "Emissions|F-Gases", 
    "Emissions|CO2|Industrial Processes", 
    "Emissions|CO2|Energy"
], model = "REMIND*", keep=False, inplace=True)

## Saving in results_v13

In [45]:
npe_results_hist.to_csv(NPE_PATH /  f"3_Data/results_v13staging/{file_suffix}_dscale_remind_complete.csv")